# Job Listings Explorer
A quick look at the **Job Listings (auto-updated)** dataset: which levels, fields and skills are in demand, who offers pay, and how long postings stay open.
Source and code: https://github.com/chandranshB/roleBase

In [ ]:
import glob
import pandas as pd
path = glob.glob("/kaggle/input/**/jobs.csv", recursive=True)[0]  # mount point differs between Kaggle versions
df = pd.read_csv(path, parse_dates=["posted", "first_seen", "closed_at"])
open_jobs = df[df.closed_at.isna()]
print(f"{len(df):,} rows, {len(open_jobs):,} open, {df.company.nunique()} companies")
df.head()

## Level and field

In [ ]:
open_jobs.level.value_counts().sort_values().plot.barh(title="Open jobs by level");

In [ ]:
open_jobs.category.value_counts().sort_values().plot.barh(title="Open jobs by field", figsize=(6, 5));

## Pay (only where the posting states a yearly range, USD)

In [ ]:
pay = open_jobs[open_jobs.pay_currency == "USD"].assign(mid=lambda d: (d.pay_min + d.pay_max) / 2)
print(f"{len(pay):,} jobs with stated USD pay")
pay.groupby("category").mid.median().sort_values().plot.barh(title="Median stated pay by field (USD)", figsize=(6, 5));

## Remote share and experience asked

In [ ]:
open_jobs.groupby("category").remote.mean().sort_values().plot.barh(title="Share of remote jobs by field", figsize=(6, 5));

In [ ]:
open_jobs.years_experience.dropna().value_counts().sort_index().plot.bar(title="Minimum years of experience asked (when stated)");

## Most requested skills

In [ ]:
skills = open_jobs.skills.dropna().str.split(";").explode()
skills.value_counts().head(20).sort_values().plot.barh(title="Top skills in open jobs", figsize=(6, 6));

## How long do postings stay open?
Only jobs that closed since tracking began, so this is biased towards short-lived postings and grows more reliable over time.

In [ ]:
closed = df.dropna(subset=["closed_at"])
days = (closed.closed_at - closed.first_seen).dt.days
print(f"{len(closed):,} closed jobs; median {days.median():.0f} days open")
days.clip(upper=60).plot.hist(bins=30, title="Days a posting stayed open (capped at 60)");